In [518]:
%reset -f 
# resetting stored variables in case there's something weird cached
from build123d import *
from ocp_vscode import *
import cadquery as cq
import time
import math
from library.tools import *
import sys
from dataclasses import dataclass, field
import bd_warehouse.thread, bd_warehouse.fastener 
from pathlib import Path
from sympy import false
from casadi import diag
from build123d.topology.composite import Part

ALL UNITS IN MM
DON'T @ ME

In [519]:
%reload_ext ocp_vscode
%load_ext autoreload
%autoreload 
print(f"Python executable: {sys.executable}")
print(f"OCP-vscode location: {sys.modules.get('ocp_vscode', 'Not found')}")
reset_show()
startTime = time.perf_counter()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Python executable: c:\Users\Kaoti\parthenon\.venv\Scripts\python.exe
OCP-vscode location: <module 'ocp_vscode' from 'c:\\Users\\Kaoti\\parthenon\\.venv\\Lib\\site-packages\\ocp_vscode\\__init__.py'>


In [ ]:
# INPUTS GO HERE


# Chamber identity
import path
chamberIdentity = "GoliathPosterior" # Other possibilities: GoliathAnterior, MalachiRight, MalachiLeft. More TBA

# Just for the moment we are presuming a universal x value, so...
# at xValue, yOffset = (2,3) we start getting issues with geometry overlapping in funny ways. Make nubs smaller?
# Same as above at (2,-4)
xValue = 0 # Domain: -5 to +5 mm
yOffset = 0 # Range: -3, -2, -1, 0, 1, 2, 3...
iteratorOffset = 5 #5 works, 6 works, 4 does not

"""
Original y values, 1-4: 7.7, 2.475, -1.775, -7
"""

# Coordinate sites / Penetration center points - these will be inputs which generate whole structure
penetration1 = (xValue, (iteratorOffset * 1.5)+ yOffset)
penetration2 = (xValue, (iteratorOffset * 0.5) + yOffset)
penetration3 = (xValue, (-iteratorOffset * 0.5) + yOffset)
penetration4 = (xValue, (-iteratorOffset * 1.5) + yOffset)
points2D = (penetration1, penetration2, penetration3, penetration4)

# Diagnostic mode yes or no? If y, then we should set it up to have Show() commands for diagnostic purposes that turn on with a time.sleep() so we can do analysis and show off what's happening. 
diagnosticMode = 1 # user input field eventually. 0 is no, 1 is full diagnostics, 2 is timefield reporting for subfield functions only. 
smallDiagnosticTime = 0.1
largeDiagnosticTime = 0.5

# Do you want to save this file? 0 is no, 1 is yes. Needs a file name; this will be used as a suffix. 
fileName = "ParametricGuideTubeFrame"
saveyn = 0

In [521]:
# Calling chamberCylinder to construct the basisCylinder

basisCylinder = chamberCylinder(chamberIdentity,diagnosticMode)
# note: 10% of time

4.241545500000939


In [522]:
# Calling meshPlanes to get our starting extrusion planes, ending extrusion solids, and 3D coordinates.

# Below are some parameters which govern the GT shaft dimensions. Easy permutation layer for later designs if needed.
shaftHeight = 3.85 # This is a rough estimation based on Anna's onshape
shaftDiameter = 4
innerShaftDiameter = 0.675000
zOffset = 2

points3D, bottomSurfaceSolids, startingOffsetPlanes = meshPlanes(chamberIdentity=chamberIdentity, points2D=points2D, 
                                                                 ZOffset = zOffset, shaftHeight = shaftHeight, shaftDiameter = shaftDiameter, 
                                                                 diagnosticMode = diagnosticMode, largeDiagnosticTime = 0.5, smallDiagnosticTime = 0.10)

Pseudocode: chamberCylinder, meshPlanes, siteShaft, siteNub, siteLoft, armConstructor, frameMerger, saveMyFile

In [523]:
# Calling siteShaft to construct shafts

shaftListWithThroughHolesFilletedTwice = siteShaft(startingOffsetPlanes, bottomSurfaceSolids, innerShaftDiameter = innerShaftDiameter, 
                                                   shaftDiameter = shaftDiameter, diagnosticMode = diagnosticMode, largeDiagnosticTime = largeDiagnosticTime, 
                                                   smallDiagnosticTime = smallDiagnosticTime)


In [524]:
# GT Depth Calculator: To-do?

In [525]:
nubsList, nubTemplatesFlattened = nubConstructor(startingOffsetPlanes, diagnosticMode = diagnosticMode, largeDiagnosticTime = largeDiagnosticTime, 
                                                   smallDiagnosticTime = smallDiagnosticTime)

In [526]:
# loftConstructor function
# inputs: nubTemplatesFlattened, nubsList

overlaps = []

# Overlap areas between nubs 2,4
overlaps.append(nubTemplatesFlattened[2] & nubTemplatesFlattened[4])

# Overlap areas between nubs 6,8
overlaps.append(nubTemplatesFlattened[6] & nubTemplatesFlattened[8])

# Overlap areas between nubs 8,10
overlaps.append(nubTemplatesFlattened[10] & nubTemplatesFlattened[12])


# Constructing overlap solids, pruning union parts between different shafts

overlapSolids = []
prunedParts = []
lofts = []



# Generating overlap structures
for i, x in enumerate(overlaps):
    with BuildPart() as firstpart:
        extrude(overlaps[i], amount = shaftHeight*10, dir = (0,0,-1))
        overlapSolids.append(firstpart.part)

# Deleting overlap between shafts 1 and 2, numbered from +y to -y in global coordinates
with BuildPart() as pt:
    add (nubsList[2])
    if overlapSolids[0] is not None:
        add (overlapSolids[0], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

with BuildPart() as pt:
    add (nubsList[4])
    if overlapSolids[0] is not None:
        add (overlapSolids[0], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

loft1 = loftMe(nubsList[2], nubsList[4])
lofts.append(loft1)

# Deleting overlap between shafts 10 and 12, numbered from +y to -y in global coordinates
with BuildPart() as pt:
    add (nubsList[10])
    if overlapSolids[2] is not None:
        add (overlapSolids[2], mode = Mode.SUBTRACT)
        
    prunedParts.append(pt.part)

with BuildPart() as pt:
    add (nubsList[12])
    if overlapSolids[2] is not None:
        add (overlapSolids[2], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

loft2 = loftMe(nubsList[10], nubsList[12])
lofts.append(loft2)

with BuildPart() as pt:
    if startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
        add (nubsList[6])
    elif startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
        add (nubsList[8])
    else:
        add (nubsList[6])
        add (nubsList[8])
    if overlapSolids[1] is not None:
        add(overlapSolids[1], mode=Mode.SUBTRACT)

    prunedParts.append(pt.part)
    
loft3 = loftMe(nubsList[6], nubsList[8])

if diagnosticMode == 1: 
    show(prunedParts, colors=["#e8b024", "#e8b024", "#e8b024",  "lightblue"])
    len(prunedParts)
    time.sleep(largeDiagnosticTime)


In [527]:
# Combining nubs and shafts with throughholes

nubulousShaftsWithThroughHoles = []
iteratorNumbers = []

for i, x in enumerate(shaftListWithThroughHolesFilletedTwice):
    with BuildPart() as NubulousShaftWithThroughHole:
        for k, j in enumerate(nubsList):
            iteratorNumber = k // 4
            if iteratorNumber == i and j != 6 and j != 8:
                add(nubsList[k])
            elif iteratorNumber == i and j == 6 or iteratorNumber == i and j == 8:
                if startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
                    add (nubsList[6])
                elif startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
                    add (nubsList[8])
                else:
                    add (nubsList[6])
                    add (nubsList[8])
            else:
                continue
        add(shaftListWithThroughHolesFilletedTwice[i])
        add([s for s in overlapSolids if s is not None], mode = Mode.SUBTRACT)
    nubulousShaftsWithThroughHoles.append(NubulousShaftWithThroughHole.part)


if diagnosticMode == 1: 
    show(nubulousShaftsWithThroughHoles)

In [528]:
# type Part = build123d.topology.composite.Part

In [529]:

if diagnosticMode == 1: 
    print(type(nubsList[1]))

In [530]:
# Generating positional arms to connect nubs and basis cylinder

# Initializing and resetting lists
testArms = []
outputArmStream = []
handPickedIndicies = [1,5,9,13]
refinedNubsList = [nubsList[i] for i in handPickedIndicies]
mirrorNubsList = [nubsList[i + 2] for i in handPickedIndicies]

# Calls armConstructor for each site, although this is before sites are a thing. Perhaps reverse this order?
for i, x in enumerate(refinedNubsList):
    arm1, mirrorArm,armTest = armConstructor(i, refinedNubsList[i],startingOffsetPlanes[i], mirrorNubsList[i], diagnosticMode = diagnosticMode)
    outputArmStream.append(arm1)
    outputArmStream.append(mirrorArm)
    testArms.append(armTest)
    # Ensure construction sketches are either nub-centric, or somehow related directionally to the rotation of the arbor centerline. 

# Visual diagnostics for this step
if diagnosticMode == 1:
    show(basisCylinder, shaftListWithThroughHolesFilletedTwice, nubsList, outputArmStream, colors=["#e8b024", "#e8b024", "#e8b024",  "lightblue", "pink", "pink"])



In [531]:
# Site-based data storage 

# Initialize the class and contents
@dataclass
class Site:
    shaft: Part
    plane: Plane
    nubs: list
    arms: list = field(default_factory = list)
    outer: object = None
    throughHole: object = None
    combined: object = None

# Initialize and reset the variables used in this cell
sites = []
planeHeightIndicator = 0

# Adds the x-axial nubs, the outer union parts, and the shafts to their site. 
for i, plane in enumerate(startingOffsetPlanes):
    sites.append(Site(
        shaft = shaftListWithThroughHolesFilletedTwice[i],
        plane = plane,
        nubs = nubsList[i*4+1:i*4 + 4:2] + [prunedParts[i]],
        arms = outputArmStream[i*2:i*2 + 2]
    )) 

# Picks the highest(z) starting offset plane of the two middle planes. 
# Adds the central nub of the higher z site and appends to nubs
# trims the central nub of the lower z site and appends that to nubs
if startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
    sites[1].nubs.append((nubsList[6]))
    sites[2].nubs.append((prunedParts[-1]))
    planeHeightIndicator = 2 # this keeps track of the higher plane

elif startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
    sites[2].nubs.append((nubsList[8]))
    sites[1].nubs.append((prunedParts[-1]))
    planeHeightIndicator = 1 # this keeps track of the higher plane

# covers the case in which the zaxis is the same, in which case we just add both nubs
else:
    sites[1].nubs.append((nubsList[6]))
    sites[2].nubs.append((nubsList[8]))

# Visual diagnostics for this step
if diagnosticMode == 1 or diagnosticMode == 1: 
    show(*[site.shaft for site in sites], [site.nubs for site in sites], [site.arms for site in sites], basisCylinder)

In [532]:
# Sitewise construction using dataclasses

# Initializing and resetting parameters
fillets = []
filletedItems = []

# Iterate through sites and fuse them with their nubs. Then fillet.
for i, site in enumerate(sites): # Iterates
    with BuildPart() as combinedSite: 
        # Add sitewise preconstructed parts
        add(site.shaft)
        add(site.nubs)

    # Grab edges for filleting
    filletMe = new_edges(
        site.shaft, 
        *site.nubs, 
        combined = combinedSite.part).filter_by(GeomType.LINE)

    # Add those edges to a list for storage
    fillets.extend(filletMe)

    # Fillet these, using a structure which adapts to unanticipated changes in radius. If this starts erroring maybe increase max iterations.
    for idx, e in enumerate(filletMe):
        try:
            # Find fillet radius for selected edges
            r = combinedSite.part.max_fillet(filletMe, max_iterations=25)
            if diagnosticMode == 1:
                print(f"edge {idx}: ok, max radius = {r}")
            filletParameter = min((r), 0.25)

            # Fillet the thing using the edges found above
            filletedItem = fillet(filletMe, radius = filletParameter)

        # This will throw an error if the above doesn't work.
        except Exception as err:
            if diagnosticMode == 1 or diagnosticMode == 1:
                print(f"edge {idx}: BAD — {type(err).__name__}: {err}")

    # Append these to list for storage
    filletedItems.append(filletedItem)


# Diagnostics for this step will show items on ocpviewer and print the length of the list containing successfully merged items. 
if diagnosticMode == 1 or diagnosticMode == 1:
    show(filletedItems)
    print(len(filletedItems))

KeyboardInterrupt: 

In [ ]:
# Adding arms in a sitewise fashion.

# Initializing and resetting parameters
combinedSitesWithArms = []

# Iterating through sites and adding arms for each of them. 
# Will hopefully support any number of sites, but this is untested.
for i, site in enumerate(sites): # Iterates
    with BuildPart() as combinedSiteWithArms:
        # Adding together previously constructed parts
        add(filletedItems[i]) # Previous BuildPart() construct
        add(site.arms) # sitewise arm construction
        combinedSitesWithArms.append(combinedSiteWithArms.part) # Add to a list

# Diagnostics for visualizing the arms, sites, and basisCylinder in ocpviewer.
if diagnosticMode == 1: 
    show([combinedSitesWithArms for site in sites], basisCylinder)


In [ ]:
# Adding together the basisCylinder from chamberCylinder and the sitewise arms. 

# Initializing and resetting parameters
armFillets = []
filletParameter2 = 0

# This combines the basis cylinder constructed with chamberCylinder
with BuildPart() as newPart:
    # Add preconstructed parts first
    add(basisCylinder) # Add basisCylinder from chamberCylinder function
    add(combinedSitesWithArms) # Add list of sites with arms

    # Now we grab edges from the combined part
    armFillet = new_edges(
        *combinedSitesWithArms,
        basisCylinder,
        combined = newPart.part).filter_by(GeomType.LINE).filter_by(lambda e: e.length >= 2.25)

    # Now we fillet those edges. We do so by the min of 0.25mm or the largest radius the smallest intersection can sustain
    try:
        r = newPart.part.max_fillet(armFillet, max_iterations=25) # Pulls the largest r that all edges can support
        if diagnosticMode == 1: # Internal diagnostics
            print(f"edge {idx}: ok, max radius = {r}")
        filletParameter2 = min((r), 0.25) # Finds min between static and max_fillet dynamic radii
        newestPart = fillet(armFillet, radius = filletParameter2) # Constructs filleted part

    # if the fillets don't work it will throw this error in diagnostic mode. 
    except Exception as err:
        if diagnosticMode == 1:
            print(f"edge {idx}: BAD — {type(err).__name__}: {err}")

# Diagnostics for the fillets, if they don't work this will display things including the length of armFillet and the visuals in ocpviewer.
if diagnosticMode == 1 or diagnosticMode == 1: 
    armFillet[2].length
    show(*[armFillets], newestPart)
    show(basisCylinder, combinedSitesWithArms,armFillet)

In [ ]:
# Add lofts, construct finalPart, which is what we export into an STL.

# Initializing parameters
loftFillets = [] # Zeros out the list in case it's being rerun. Also initializes.
filletParameter3 = 0 # Zeros out list for rerunning purposes

# Constructing finalPart
with BuildPart() as nextPart:
    add (newestPart) # This arises from the previously called buildPart function
    add (loft1) # Lofting between nubs 2 and 4
    add (loft2) # Lofting between nubs 10 and 12
    if overlapSolids[1] is None: # This is making sure that there's no overlap; if there is overlap, earlier functionality will apply.
        add(loft3) # Lofting between central nubs

    # Now that we've added all the solid parts, we're going to grab all the edges we need to fillet from those parts.
    loftFillet = nextPart.edges().filter_by(
        GeomType.LINE).filter_by(
        lambda e: abs(e.length - 3) < 1e-6).filter_by(
        lambda e: abs(e.position_at(0).Z - e.position_at(1).Z) < 1e-6).filter_by(
        lambda e: abs(e.position_at(0).Y - e.position_at(1).Y) < 1e-6).filter_by(
        lambda e: e.center().Z < -1)

    # Then we extend a list of fillets. This is mostly just a storage mechanism so we can access this later if we have to.
    loftFillets.extend(loftFillet)

    # Now we're going to fillet these parts. We'll use the minimum of either the max_fillet value for the smallest edge or 0.25mm.
    try:
        r = nextPart.part.max_fillet(loftFillet, max_iterations=25) # Establishing the radius
        if diagnosticMode == 1: # Internal diagnostics
            print(f"edge {idx}: ok, max radius = {r}")
        filletParameter3 = min((r), 0.25) # Finding the min between the static radius and the max radius
        finalPart = (fillet(loftFillet, radius = filletParameter3)) # Saving the final part. 

    # if that doesn't work, this prints in diagnostic mode.
    except Exception as err:
        if diagnosticMode == 1:
            print(f"edge {idx}: BAD — {type(err).__name__}: {err}")

# This tests whether different edges have the capability to be filleted, and if so, their max radius. It also prints those fillets.
if diagnosticMode == 1: 
    print(loftFillets) # This prints the list.

    # This iterates through the fillets - use this if your fillets are failing, it can tell you which one is not functioning. You might have to paste it into another cell.
    for i, e in enumerate(loftFillet):
        try:
            r = nextPart.part.max_fillet([e])
            print(i, "ok — max radius:", r)
        # If the edges don't function this will alert us
        except Exception as err:
            print(i, "BAD EDGE:", type(err).__name__, err)
    print(len(nextPart.part.solids()))


show(finalPart) # Shows the final part we've now finished constructing in OCP viewer.

c


In [ ]:
# Joining file names together and saving file

# First I'm joining the coordinate names together for a uniquely identifiable string
prefix = "-".join(str(p) for p in points2D)

# Then we join that string together with the user-inputted file name, and give it an stl suffix
joinMe = [prefix, fileName]
fileNameReal = "-".join(joinMe)
fileNameActual = Path(fileNameReal + ".stl")

# Diagnostics if desired, to ensure file name is correct
if diagnosticMode == 1:
    print(prefix)
    print(fileNameReal)
    print(fileNameActual)

# Time related diagnostics
if diagnosticMode == 1 or diagnosticMode == 2: 
    elapsedTime = time.perf_counter() - startTime
    print(elapsedTime)

# as of 9/2/2026:
# diagnosticMode = 0 is who knows how long because it doesn't print! 
# diagnosticMode = 1 is 61. 1 seconds
# diagnosticMode = 2 is 26.3 seconds, 25.5 after activating the diagnostic toggle on all show() commands. 17.3 after refining the filleting to be out of a for loop.


# Saves file with composite file name if saving is turned on
if saveyn == 1:
    export_stl(finalPart, fileNameActual)

29.70570900000166
